In [ ]:
import numpy as np
import pandas as pd
from casadi import *
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from Functions.Utils import *
from Functions.Graphs import *
from scipy.interpolate import CubicSpline
from sklearn.metrics import root_mean_squared_error as RMSE

In [ ]:


def TurningSpeed(u_control_1, V_cruising, V_turning):
    """
    Calcula a velocidade de referência adaptativa com base no ângulo de esterçamento.
    
    Parâmetros:
    - u_control_1: float, ângulo atual das rodas em graus [-30, 30]
    - V_cruising: float, velocidade de cruzeiro em linha reta (ex: 20 m/s)
    - V_turning: float, velocidade reduzida limite para curvas acentuadas (ex: 8 m/s)
    
    Retorna:
    - V_ref: float, velocidade ótima calculada
    """
    # 1. Trabalha com o valor absoluto para garantir comportamento simétrico
    angulo_abs = abs(u_control_1)
    
    # 2. Define os pontos de quebra da interpolação baseados no módulo do ângulo:
    # De 0° até 3°: Retorna V_cruising
    # De 3° até 30°: Reduz proporcionalmente até V_turning
    pontos_angulo = np.linspace(0,30,2000)
    pontos_velocidade = np.flip(np.linspace(V_turning, V_cruising, 2000))
    
    # 3. Interpola linearmente de forma segura e blinda contra inputs fora do escopo
    V_ref = np.interp(angulo_abs, pontos_angulo, pontos_velocidade)
    
    return float(V_ref)

In [ ]:
# %%
size = 1
path = r'DyntheticDataset\RaceTrack4.csv'
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:] * size
y_mid = df['y_coords'].values[:] * size

# Ensure the track loops closed cleanly
if not np.allclose([x_mid[0], y_mid[0]], [x_mid[-1], y_mid[-1]]):
    x_mid = np.append(x_mid, x_mid[0])
    y_mid = np.append(y_mid, y_mid[0])

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

# Calculate cumulative distance (arc length 's') along track coordinates
dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

# Linear interpolators to find exact continuous (x, y) coordinates for any distance s
get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

In [ ]:
# %% [markdown]
# 2. CASADI OPTI PROBLEM CONFIGURATION (Defined ONCE outside loop)

# %%
dt = 0.1
sim_steps = 400
V_cruising = 16
V_max = 25
V_turning = 8
n_horizon = 20
track_percentual = 0.9
l = 2.5 # Wheelbase

opti = Opti()

# Decision variables over the prediction horizon
X = opti.variable(4, n_horizon + 1) # States: [x_pos, y_pos, v, psi]
U = opti.variable(2, n_horizon)     # Controls: [a, delta]

# Parameters
X0 = opti.parameter(4)
X_REF = opti.parameter(n_horizon)
Y_REF = opti.parameter(n_horizon)
V_REF = opti.parameter(n_horizon)
U_PREV = opti.parameter(2)  

W_X = 1
W_Y = 1
W_speed = 5 
W_acc = 1.5
W_delta = 0.25
W_U0 = 15
W_U1 = 2

obj = 0
opti.subject_to(X[:, 0] == X0)

# Build the execution graph symbolic tree once
for k in range(n_horizon):
    x_k   = X[0, k]
    y_k   = X[1, k]
    v_k   = X[2, k]
    psi_k = X[3, k]
    
    a_k     = U[0, k]
    delta_k = U[1, k]

    beta = atan(0.5 * tan(delta_k))
    
    next_x   = x_k + dt * (v_k * cos(psi_k + beta))
    next_y   = y_k + dt * (v_k * sin(psi_k + beta))
    next_v   = v_k + dt * a_k
    next_psi = psi_k + dt * ((v_k / l) * sin(beta))

    opti.subject_to(X[0, k+1] == next_x)
    opti.subject_to(X[1, k+1] == next_y)
    opti.subject_to(X[2, k+1] == next_v)
    opti.subject_to(X[3, k+1] == next_psi)

    # Base state and input magnitudes stage cost
    obj += W_X*(x_k - X_REF[k])**2 + W_Y*(y_k - Y_REF[k])**2 + W_speed * (v_k - V_REF[k])**2
    obj += W_acc * a_k**2 + W_delta * delta_k**2

    # Slew rate penalties inside the preview horizon steps
    if k > 0:
        obj += W_U0 * (U[0, k] - U[0, k-1])**2  # Jerk penalty
        obj += W_U1 * (U[1, k] - U[1, k-1])**2   # Steering rate penalty

# Smooth transitions linking external past inputs directly to the initial horizon decision step
#obj += 25.0 * (U[0, 0] - U_PREV[0])**2
#obj += 2.0 * (U[1, 0] - U_PREV[1])**2

# Terminal cost step calculation
#obj += (X[0, n_horizon] - X_REF[n_horizon - 1])**2 + \
#       (X[1, n_horizon] - Y_REF[n_horizon - 1])**2 + \
#       W_speed * (X[2, n_horizon] - V_REF[n_horizon - 1])**2

opti.minimize(obj)

# Operational Bound limits
opti.subject_to(opti.bounded(-7.0, U[0, :], 3))                 
opti.subject_to(opti.bounded(-np.deg2rad(30), U[1, :], np.deg2rad(30))) 
opti.subject_to(opti.bounded(0.0, X[2, :], V_cruising*1))                  

# Setup solver configuration options once
opts = {"ipopt.print_level": 0, "print_time": 0, "ipopt.warm_start_init_point": "yes"}
opti.solver('ipopt', opts)


In [ ]:
# %% [markdown]
# 3. CLOSED-LOOP SIMULATION LOOP

# %%
s_total_traveled = 0.0
last_current_idx = 0

x0_start = np.array([50, y_mid[0], 0.0, 0.0])
x_current = x0_start.copy()
current_time = 0.0

# Memory array tracking previous system inputs (Starts at 0.0 standstill condition)
u_executed_prev = np.array([0.0, 0.0])

t_history = [current_time]
x_history = [x_current[0]]
y_history = [x_current[1]]
xRef_history = [x_current[0]]
yRef_history = [x_current[1]]
v_history = [x_current[2]]
v_ref_history = [0]
v_k_history = []
v_horizon = []
acc_history = [0]
delta_history = [0]
psi_history = [0]
is_turning_sharp = False
turning = [0]

# Continuous trajectory memory for warm-starting
last_sol_X = np.tile(x_current.reshape(4, 1), (1, n_horizon + 1))
last_sol_U = np.zeros((2, n_horizon))

for step in range(sim_steps):
    if (step) % 300 == 0: print('step:', step)
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: 
        idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2:
        idx_diff -= len(x_mid)
        
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # Generate Time-Varying Horizon Reference Profiles
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * track_percentual:
            v_ref_horizon[k] = 0.0

        elif is_turning_sharp:
            speed = TurningSpeed(np.rad2deg(u_control[1]),V_cruising*0.8, V_turning)
            #v_ref_horizon[k] = TurningSpeed(np.rad2deg(u_control[1]),V_cruising, V_turning)
            v_ref_horizon[k] = speed

        else:
            v_ref_horizon[k] = V_cruising

    # Update parameters mapping arrays
    opti.set_value(X0, x_current)
    opti.set_value(X_REF, x_ref_horizon)
    opti.set_value(Y_REF, y_ref_horizon)
    opti.set_value(V_REF, v_ref_horizon)
    opti.set_value(U_PREV, u_executed_prev)  # Push historical tracking input directly to solver memory

    # Seed guesses
    opti.set_initial(X, last_sol_X)
    opti.set_initial(U, last_sol_U)

    try:
        sol = opti.solve()
        u_control = sol.value(U[:, 0])
        last_sol_X = sol.value(X)
        last_sol_U = sol.value(U)
    except Exception as e:
        # Fallback sub-optimal state capture strategy
        u_control = opti.debug.value(U[:, 0])
        last_sol_X = opti.debug.value(X)
        last_sol_U = opti.debug.value(U)

    # Save the selected control inputs to match with the next tracking frame interval
    u_executed_prev = u_control.copy()

    # --- Plant Simulator (Process Step using Forward Euler) ---
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))

    is_turning_sharp = (u_control[1] >= np.deg2rad(3)) or (u_control[1] <= -np.deg2rad(3))
    turning.append(is_turning_sharp)
    
    x_current = np.array([x_next, y_next, v_next, psi_next])
    current_time += dt

    t_history.append(current_time)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    xRef_history.append(x_ref_horizon[0])
    yRef_history.append(y_ref_horizon[0])
    v_history.append(x_current[2])
    v_k_history.append(last_sol_X[2])
    v_ref_history.append(v_ref_horizon[0])
    acc_history.append(u_control[0])
    psi_history.append(x_current[3])
    delta_history.append(u_control[1])
    v_horizon.append(v_ref_horizon)

    if s_total_traveled >= track_length * track_percentual and x_current[2] < 0.1:
        print(f"Simulation completed! Lap finished and vehicle stopped on centerline at step {step}.")
        break

# Convert historical data into numpy arrays for rendering 
t_history = np.array(t_history)
x_history = np.array(x_history)
y_history = np.array(y_history)
xRef_history = np.array(xRef_history)
yRef_history = np.array(yRef_history)
v_history = np.array(v_history)
v_ref_history = np.array(v_ref_history)
acc_history = np.array(acc_history)
delta_history = np.array(delta_history)
delta_history = np.rad2deg(delta_history)

psi_history = np.array(psi_history)
psi_history = np.rad2deg(psi_history)
turning = [0 if t==False else 1 for t in turning]
turning = np.array(turning)



In [ ]:
ySeries=[v_ref_history, v_history, acc_history, delta_history, turning]
xSeries=[t_history for i in range(len(ySeries))]
names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
         'Acceleration (m/s²)', 'Stirring Angle (deg)', 'Truning']
PlotSeriesPLY(xSeries,ySeries,names,xLabel='Time (s)',title='Parameters Over Time')
data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]
PlotMPCTracksPLY(data,width=800,height=500)

In [ ]:
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go

# --- Adaptive Turning Speed Function ---
def TurningSpeed(u_control_1, V_cruising, V_turning):
    angulo_abs = abs(u_control_1)
    pontos_angulo = np.linspace(0, 30, 2000)
    pontos_velocidade = np.flip(np.linspace(V_turning, V_cruising, 2000))
    V_ref = np.interp(angulo_abs, pontos_angulo, pontos_velocidade)
    return float(V_ref)

# --- Linearized Kinematic Bicycle Model Matrices ---
def get_linear_model_matrices(x_nominal, u_nominal, dt, l=2.5):
    """
    Returns the linearized state-space matrices A and B such that:
    x_{k+1} = A*x_k + B*u_k + c
    States: [x, y, v, psi]
    Controls: [a, delta]
    """
    _, _, v, psi = x_nominal
    a, delta = u_nominal
    
    beta = np.arctan(0.5 * np.tan(delta))
    
    # Cosine/sine derivatives evaluations
    cos_psi_beta = np.cos(psi + beta)
    sin_psi_beta = np.sin(psi + beta)
    
    # Partial derivatives for A matrix (Jacobian w.r.t State)
    # dx_next/dx = 1, dx_next/dy = 0, dx_next/dv = dt*cos, dx_next/dpsi = -dt*v*sin
    A = np.array([
        [1.0, 0.0,  dt * cos_psi_beta, -dt * v * sin_psi_beta],
        [0.0, 1.0,  dt * sin_psi_beta,  dt * v * cos_psi_beta],
        [0.0, 0.0,                1.0,                    0.0],
        [0.0, 0.0, dt * np.sin(beta)/l,                   1.0]
    ])
    
    # Partial derivatives w.r.t delta (Chain rule via beta)
    dbeta_ddelta = 0.5 * (1 + np.tan(delta)**2) / (1 + (0.5 * np.tan(delta))**2)
    dx_ddelta = -dt * v * sin_psi_beta * dbeta_ddelta
    dy_ddelta = dt * v * cos_psi_beta * dbeta_ddelta
    dpsi_ddelta = dt * (v / l) * np.cos(beta) * dbeta_ddelta
    
    # B matrix (Jacobian w.r.t Controls)
    B = np.array([
        [0.0, dx_ddelta],
        [0.0, dy_ddelta],
        [dt,  0.0],
        [0.0, dpsi_ddelta]
    ])
    
    # Affine constant term to offset linearization point mismatch
    x_next_nominal = np.array([
        x_nominal[0] + dt * v * cos_psi_beta,
        x_nominal[1] + dt * v * sin_psi_beta,
        v + dt * a,
        psi + dt * (v / l) * np.sin(beta)
    ])
    c = x_next_nominal - A @ x_nominal - B @ u_nominal
    
    return A, B, c

# --- Track Data Loading & Setup ---
size = 1
# Update path matching your local setup
path = r'DyntheticDataset\RaceTrack4.csv' 
df = pd.read_csv(path)
x_mid = df['x_coords'].values[:] * size
y_mid = df['y_coords'].values[:] * size

track_points = np.vstack((x_mid, y_mid)).T
track_tree = KDTree(track_points)

dx = np.diff(x_mid)
dy = np.diff(y_mid)
segment_lengths = np.sqrt(dx**2 + dy**2)
s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
track_length = s_coor[-1]

get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

# --- MPC Simulation Configuration ---
dt = 0.25
sim_steps = 800
V_cruising = 16
V_turning = 8
n_horizon = 20
track_percentual = 1
l = 2.5 

# Cost weights
W_X, W_Y, W_speed = 1.0, 1.0, 0.01
W_acc, W_delta = 1.5, 0.25
W_U0, W_U1 = 1, 2.0  # Slew-rate/Jerk weights

# Initialization
s_total_traveled = 0.0
last_current_idx = 0
x_current = np.array([50, y_mid[0], 0.0, 0.0])
u_prev = np.array([0.0, 0.0])

# History collections
t_history = [0.0]
x_history = [x_current[0]]
y_history = [x_current[1]]
v_history = [x_current[2]]
v_ref_history = [0.0]
acc_history = [0.0]
delta_history = [0.0]
turning_history = [0]
psi_history = [0]

is_turning_sharp = False

# --- Closed Loop Simulation Loop ---
for step in range(sim_steps):
    if step % 50 == 0: 
        print(f'Step: {step} | Speed: {x_current[2]:.2f} m/s | Distance traveled: {s_total_traveled:.2f} / {track_length:.2f} m')
        
    _, current_idx = track_tree.query([x_current[0], x_current[1]])
    
    idx_diff = current_idx - last_current_idx
    if idx_diff < -len(x_mid)/2: idx_diff += len(x_mid)
    elif idx_diff > len(x_mid)/2: idx_diff -= len(x_mid)
    if idx_diff > 0:
        s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
    last_current_idx = current_idx

    # References Profiles generation
    s_projected = s_coor[current_idx]
    x_ref_horizon = np.zeros(n_horizon)
    y_ref_horizon = np.zeros(n_horizon)
    v_ref_horizon = np.zeros(n_horizon)

    for k in range(n_horizon):
        s_projected += max(x_current[2], 1.5) * dt 
        s_wrapped = s_projected % track_length
        x_ref_horizon[k] = get_x_at_s(s_wrapped)
        y_ref_horizon[k] = get_y_at_s(s_wrapped)
        
        if s_total_traveled >= track_length * track_percentual:
            v_ref_horizon[k] = 0.0
        elif is_turning_sharp:
            v_ref_horizon[k] = TurningSpeed(np.rad2deg(u_prev[1]), V_cruising * 0.8, V_turning)
        else:
            v_ref_horizon[k] = V_cruising

    # --- Setup and Solve CVXPY Conic Problem via MOSEK ---
    X_cvx = cp.Variable((4, n_horizon + 1))
    U_cvx = cp.Variable((2, n_horizon))
    
    cost = 0
    constraints = [X_cvx[:, 0] == x_current]
    
    # Sequential Linearization around the current active state/input as the baseline nominal guess
    A_lin, B_lin, c_lin = get_linear_model_matrices(x_current, u_prev, dt, l)
    
    for k in range(n_horizon):
        # State dynamics (LTV formulation)
        constraints += [X_cvx[:, k+1] == A_lin @ X_cvx[:, k] + B_lin @ U_cvx[:, k] + c_lin]
        
        # State & Control Magnitude Tracking Objective Costs
        cost += W_X * cp.square(X_cvx[0, k] - x_ref_horizon[k])
        cost += W_Y * cp.square(X_cvx[1, k] - y_ref_horizon[k])
        cost += W_speed * cp.square(X_cvx[2, k] - v_ref_horizon[k])
        cost += W_acc * cp.square(U_cvx[0, k])
        cost += W_delta * cp.square(U_cvx[1, k])
        
        # Smooth Input Changes (Slew rates limits)
        if k == 0:
            cost += W_U0 * cp.square(U_cvx[0, 0] - u_prev[0])
            cost += W_U1 * cp.square(U_cvx[1, 0] - u_prev[1])
        else:
            cost += W_U0 * cp.square(U_cvx[0, k] - U_cvx[0, k-1])
            cost += W_U1 * cp.square(U_cvx[1, k] - U_cvx[1, k-1])
            
        # Hard Operational Boundary Constraints
        constraints += [U_cvx[0, k] >= -7.0, U_cvx[0, k] <= 3.0]
        constraints += [U_cvx[1, k] >= -np.deg2rad(30), U_cvx[1, k] <= np.deg2rad(30)]
        constraints += [X_cvx[2, k] >= 0.0, X_cvx[2, k] <= V_cruising]

    # Explicitly compile and call the MOSEK Solver engine
    prob = cp.Problem(cp.Minimize(cost), constraints)
    try:
        prob.solve(solver=cp.MOSEK, verbose=False)
        u_control = U_cvx[:, 0].value
        # Fail safe check for optimizer errors
        if u_control is None:
            raise ValueError
    except Exception:
        # Fallback to zeroed inputs if solver encounters an issue
        u_control = np.array([0.0, u_prev[1]])

    # --- Plant Simulator (Process Forward Euler Update) ---
    beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
    x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
    y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
    v_next = x_current[2] + dt * u_control[0]
    psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))
    
    is_turning_sharp = abs(u_control[1]) >= np.deg2rad(3)
    x_current = np.array([x_next, y_next, v_next, psi_next])
    u_prev = u_control.copy()
    
    # Log telemetry
    t_history.append((step + 1) * dt)
    x_history.append(x_current[0])
    y_history.append(x_current[1])
    v_history.append(x_current[2])
    v_ref_history.append(v_ref_horizon[0])
    acc_history.append(u_control[0])
    delta_history.append(np.rad2deg(u_control[1]))
    psi_history.append(np.rad2deg(x_current[3]))
    turning_history.append(1 if is_turning_sharp else 0)

    if s_total_traveled >= track_length * track_percentual and x_current[2] < 0.25:
        print(f"Simulation completed cleanly! Finished track at step {step}.")
        break

    if s_total_traveled >= track_length * 1.25:
        print(f"Simulation completed cleanly! Finished track at step {step}.")
        break


In [ ]:
ySeries=[v_ref_history, v_history, acc_history, delta_history, turning_history]
xSeries=[t_history for i in range(len(ySeries))]
names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
         'Acceleration (m/s²)', 'Stirring Angle (deg)', 'Truning']
data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]

PlotSeriesPLY(xSeries,ySeries,names,xLabel='Time (s)',title='Parameters Over Time')
PlotMPCTracksPLY(data,width=800,height=500)

In [ ]:

def SimulateRT(dt=0.25,n_horizon = 20,sim_steps = 800, BreakCheck = False,
               W_X = 1,W_Y = 1,W_speed = 0.01 ,W_acc = 1.5,W_delta = 0.25,W_U0 = 1,W_U1 = 2,
               size=1,show=False):
    # --- Track Data Loading & Setup ---
    # Update path matching your local setup
    path = r'DyntheticDataset\RaceTrack4.csv' 
    df = pd.read_csv(path)
    x_mid = df['x_coords'].values[:] * size
    y_mid = df['y_coords'].values[:] * size

    track_points = np.vstack((x_mid, y_mid)).T
    track_tree = KDTree(track_points)

    dx = np.diff(x_mid)
    dy = np.diff(y_mid)
    segment_lengths = np.sqrt(dx**2 + dy**2)
    s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
    track_length = s_coor[-1]

    get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
    get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

    # --- MPC Simulation Configuration ---
    V_cruising = 16
    V_turning = 8
    track_percentual = 1
    l = 2.5 

    # Initialization
    s_total_traveled = 0.0
    last_current_idx = 0
    x_current = np.array([50*size, y_mid[0], 0.0, 0.0])
    u_prev = np.array([0.0, 0.0])

    # History collections
    t_history = [0.0]
    x_history = [x_current[0]]
    y_history = [x_current[1]]
    v_history = [x_current[2]]
    v_ref_history = [0.0]
    acc_history = [0.0]
    delta_history = [0.0]
    turning_history = [0]
    psi_history = [0]

    is_turning_sharp = False

    # --- Closed Loop Simulation Loop ---
    for step in range(sim_steps):
        if step % 50 == 0 and show: 
            print(f'Step: {step} | Speed: {x_current[2]:.2f} m/s | Distance traveled: {s_total_traveled:.2f} / {track_length:.2f} m')
            
        _, current_idx = track_tree.query([x_current[0], x_current[1]])
        
        idx_diff = current_idx - last_current_idx
        if idx_diff < -len(x_mid)/2: idx_diff += len(x_mid)
        elif idx_diff > len(x_mid)/2: idx_diff -= len(x_mid)
        if idx_diff > 0:
            s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
        last_current_idx = current_idx

        # References Profiles generation
        s_projected = s_coor[current_idx]
        x_ref_horizon = np.zeros(n_horizon)
        y_ref_horizon = np.zeros(n_horizon)
        v_ref_horizon = np.zeros(n_horizon)

        for k in range(n_horizon):
            s_projected += max(x_current[2], 1.5) * dt 
            s_wrapped = s_projected % track_length
            x_ref_horizon[k] = get_x_at_s(s_wrapped)
            y_ref_horizon[k] = get_y_at_s(s_wrapped)
            
            if s_total_traveled >= track_length * track_percentual:
                v_ref_horizon[k] = 0.0
            elif is_turning_sharp:
                v_ref_horizon[k] = TurningSpeed(np.rad2deg(u_prev[1]), V_cruising * 0.8, V_turning)
            else:
                v_ref_horizon[k] = V_cruising

        # --- Setup and Solve CVXPY Conic Problem via MOSEK ---
        X_cvx = cp.Variable((4, n_horizon + 1))
        U_cvx = cp.Variable((2, n_horizon))
        
        cost = 0
        constraints = [X_cvx[:, 0] == x_current]
        
        # Sequential Linearization around the current active state/input as the baseline nominal guess
        A_lin, B_lin, c_lin = get_linear_model_matrices(x_current, u_prev, dt, l)
        
        for k in range(n_horizon):
            # State dynamics (LTV formulation)
            constraints += [X_cvx[:, k+1] == A_lin @ X_cvx[:, k] + B_lin @ U_cvx[:, k] + c_lin]
            
            # State & Control Magnitude Tracking Objective Costs
            cost += W_X * cp.square(X_cvx[0, k] - x_ref_horizon[k])
            cost += W_Y * cp.square(X_cvx[1, k] - y_ref_horizon[k])
            cost += W_speed * cp.square(X_cvx[2, k] - v_ref_horizon[k])
            cost += W_acc * cp.square(U_cvx[0, k])
            cost += W_delta * cp.square(U_cvx[1, k])
            
            # Smooth Input Changes (Slew rates limits)
            if k == 0:
                cost += W_U0 * cp.square(U_cvx[0, 0] - u_prev[0])
                cost += W_U1 * cp.square(U_cvx[1, 0] - u_prev[1])
            else:
                cost += W_U0 * cp.square(U_cvx[0, k] - U_cvx[0, k-1])
                cost += W_U1 * cp.square(U_cvx[1, k] - U_cvx[1, k-1])
                
            # Hard Operational Boundary Constraints
            constraints += [U_cvx[0, k] >= -7.0, U_cvx[0, k] <= 3.0]
            constraints += [U_cvx[1, k] >= -np.deg2rad(30), U_cvx[1, k] <= np.deg2rad(30)]
            constraints += [X_cvx[2, k] >= 0.0, X_cvx[2, k] <= V_cruising]

        # Explicitly compile and call the MOSEK Solver engine
        prob = cp.Problem(cp.Minimize(cost), constraints)
        try:
            prob.solve(solver=cp.MOSEK, verbose=False)
            u_control = U_cvx[:, 0].value
            # Fail safe check for optimizer errors
            if u_control is None:
                raise ValueError
        except Exception:
            # Fallback to zeroed inputs if solver encounters an issue
            u_control = np.array([0.0, u_prev[1]])

        # --- Plant Simulator (Process Forward Euler Update) ---
        beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
        x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
        y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
        v_next = x_current[2] + dt * u_control[0]
        psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))
        
        is_turning_sharp = abs(u_control[1]) >= np.deg2rad(3)
        x_current = np.array([x_next, y_next, v_next, psi_next])
        u_prev = u_control.copy()
        
        # Log telemetry
        t_history.append((step + 1) * dt)
        x_history.append(x_current[0])
        y_history.append(x_current[1])
        v_history.append(x_current[2])
        v_ref_history.append(v_ref_horizon[0])
        acc_history.append(u_control[0])
        delta_history.append(np.rad2deg(u_control[1]))
        psi_history.append(np.rad2deg(x_current[3]))
        turning_history.append(1 if is_turning_sharp else 0)

        delta_check = np.abs(np.array(delta_history))
        delta_check = delta_check[delta_check>26.5]

        if s_total_traveled >= track_length * track_percentual and x_current[2] < 0.99:
            if show:
                print(f"Simulation completed cleanly! Finished track at step {step}.")
            break

        elif s_total_traveled >= track_length * 1.05:
            if show:
                print(f"Simulation completed! Broke track at step {step}.")
            break

        elif len(delta_check) > 10:
            if show:
                print(f"Broke simulation at step {step}.")
            BreakCheck = True
            break

    # Convert historical data into numpy arrays for rendering 
    t_history = np.array(t_history)
    x_history = np.array(x_history)
    y_history = np.array(y_history)
    v_history = np.array(v_history)
    v_ref_history = np.array(v_ref_history)
    acc_history = np.array(acc_history)
    delta_history = np.array(delta_history)
    psi_history = np.array(psi_history)
    turning_history.append(1 if is_turning_sharp else 0)


    score = RMSE(v_ref_history, v_history)
    ySeries=[v_ref_history, v_history, acc_history, delta_history, turning_history]
    xSeries=[t_history for i in range(len(ySeries))]
    names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
            'Acceleration (m/s²)', 'Stirring Angle (deg)', 'Truning']
    data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]

    return score,BreakCheck,[xSeries,ySeries,names,data]


In [ ]:
score,BreakCheck,[xSeries,ySeries,names,data] = SimulateRT(
    dt=0.25,n_horizon = 20,sim_steps = 800, BreakCheck = False,
    W_X = 1,W_Y = 1,W_speed = 10 ,W_acc = 1.5,W_delta = 0.25,W_U0 = 1,W_U1 = 2, size=1,show=True)

In [ ]:
ySeries=[v_ref_history, v_history, acc_history, delta_history, turning_history]
xSeries=[t_history for i in range(len(ySeries))]
names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
         'Acceleration (m/s²)', 'Stirring Angle (deg)', 'Truning']
PlotSeriesPLY(xSeries,ySeries,names,xLabel='Time (s)',title='Parameters Over Time')
data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]
PlotMPCTracksPLY(data,width=800,height=500)

In [ ]:
def SimulateRT(dt=0.25,n_horizon = 20,sim_steps = 800,
               W_X = 1,W_Y = 1,W_speed = 10 ,W_acc = 1.5,W_delta = 0.25,W_U0 = 1,W_U1 = 2,
               size=1,show=False):
    # --- Track Data Loading & Setup ---
    # Update path matching your local setup

    BreakCheck = False
    path = r'DyntheticDataset\RaceTrack4.csv' 
    df = pd.read_csv(path)
    x_mid = df['x_coords'].values[:] * size
    y_mid = df['y_coords'].values[:] * size

    track_points = np.vstack((x_mid, y_mid)).T
    track_tree = KDTree(track_points)

    dx = np.diff(x_mid)
    dy = np.diff(y_mid)
    segment_lengths = np.sqrt(dx**2 + dy**2)
    s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
    track_length = s_coor[-1]

    get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
    get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

    # --- MPC Simulation Configuration ---
    V_cruising = 16
    V_turning = 8
    track_percentual = 0.9
    l = 2.5 

    # Initialization
    s_total_traveled = 0.0
    last_current_idx = 0
    x_current = np.array([x_mid[0], y_mid[0], 0.0, 0.0])
    u_prev = np.array([0.0, 0.0])

    # History collections
    t_history = [0.0]
    x_history = [x_current[0]]
    y_history = [x_current[1]]
    v_history = [x_current[2]]
    v_ref_history = [0.0]
    acc_history = [0.0]
    delta_history = [0.0]
    turning_history = [0]
    psi_history = [0]

    is_turning_sharp = False

    # --- Closed Loop Simulation Loop ---
    for step in range(sim_steps):
        if step % 50 == 0 and show: 
            print(f'Step: {step} | Speed: {x_current[2]:.2f} m/s | Distance traveled: {s_total_traveled:.2f} / {track_length:.2f} m')
            
        _, current_idx = track_tree.query([x_current[0], x_current[1]])
        
        idx_diff = current_idx - last_current_idx
        if idx_diff < -len(x_mid)/2: idx_diff += len(x_mid)
        elif idx_diff > len(x_mid)/2: idx_diff -= len(x_mid)
        if idx_diff > 0:
            s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
        last_current_idx = current_idx

        # References Profiles generation
        s_projected = s_coor[current_idx]
        x_ref_horizon = np.zeros(n_horizon)
        y_ref_horizon = np.zeros(n_horizon)
        v_ref_horizon = np.zeros(n_horizon)

        for k in range(n_horizon):
            s_projected += max(x_current[2], 1.5) * dt 
            s_wrapped = s_projected % track_length
            x_ref_horizon[k] = get_x_at_s(s_wrapped)
            y_ref_horizon[k] = get_y_at_s(s_wrapped)
            
            if s_total_traveled >= track_length * track_percentual:
                v_ref_horizon[k] = 0.0
            elif is_turning_sharp:
                v_ref_horizon[k] = TurningSpeed(np.rad2deg(u_prev[1]), V_cruising * 0.8, V_turning)
            else:
                v_ref_horizon[k] = V_cruising

        # --- Setup and Solve CVXPY Conic Problem via MOSEK ---
        X_cvx = cp.Variable((4, n_horizon + 1))
        U_cvx = cp.Variable((2, n_horizon))
        
        cost = 0
        constraints = [X_cvx[:, 0] == x_current]
        
        # Sequential Linearization around the current active state/input as the baseline nominal guess
        A_lin, B_lin, c_lin = get_linear_model_matrices(x_current, u_prev, dt, l)
        
        for k in range(n_horizon):
            # State dynamics (LTV formulation)
            constraints += [X_cvx[:, k+1] == A_lin @ X_cvx[:, k] + B_lin @ U_cvx[:, k] + c_lin]
            
            # State & Control Magnitude Tracking Objective Costs
            cost += W_X * cp.square(X_cvx[0, k] - x_ref_horizon[k])
            cost += W_Y * cp.square(X_cvx[1, k] - y_ref_horizon[k])
            cost += W_speed * cp.square(X_cvx[2, k] - v_ref_horizon[k])
            cost += W_acc * cp.square(U_cvx[0, k])
            cost += W_delta * cp.square(U_cvx[1, k])
            
            # Smooth Input Changes (Slew rates limits)
            if k == 0:
                cost += W_U0 * cp.square(U_cvx[0, 0] - u_prev[0])
                cost += W_U1 * cp.square(U_cvx[1, 0] - u_prev[1])
            else:
                cost += W_U0 * cp.square(U_cvx[0, k] - U_cvx[0, k-1])
                cost += W_U1 * cp.square(U_cvx[1, k] - U_cvx[1, k-1])
                
            # Hard Operational Boundary Constraints
            constraints += [U_cvx[0, k] >= -7.0, U_cvx[0, k] <= 3.0]
            constraints += [U_cvx[1, k] >= -np.deg2rad(30), U_cvx[1, k] <= np.deg2rad(30)]
            constraints += [X_cvx[2, k] >= 0.0, X_cvx[2, k] <= V_cruising]

        # Explicitly compile and call the MOSEK Solver engine
        prob = cp.Problem(cp.Minimize(cost), constraints)
        try:
            prob.solve(solver=cp.MOSEK, verbose=False)
            u_control = U_cvx[:, 0].value
            # Fail safe check for optimizer errors
            if u_control is None:
                raise ValueError
        except Exception:
            # Fallback to zeroed inputs if solver encounters an issue
            u_control = np.array([0.0, u_prev[1]])

        # --- Plant Simulator (Process Forward Euler Update) ---
        beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
        x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
        y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
        v_next = x_current[2] + dt * u_control[0]
        psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))
        
        is_turning_sharp = abs(u_control[1]) >= np.deg2rad(3)
        x_current = np.array([x_next, y_next, v_next, psi_next])
        u_prev = u_control.copy()
        
        # Log telemetry
        t_history.append((step + 1) * dt)
        x_history.append(x_current[0])
        y_history.append(x_current[1])
        v_history.append(x_current[2])
        v_ref_history.append(v_ref_horizon[0])
        acc_history.append(u_control[0])
        delta_history.append(np.rad2deg(u_control[1]))
        psi_history.append(np.rad2deg(x_current[3]))
        turning_history.append(1 if is_turning_sharp else 0)

        delta_check = (np.abs(np.array(delta_history)))
        delta_check = delta_check[delta_check>26.5]

        if s_total_traveled >= track_length * track_percentual and x_current[2] < 0.5:
            if show:
                print(f"Simulation completed cleanly! Finished track at step {step}.")
            break

        elif s_total_traveled >= track_length * 1.1:
            if show:
                print(f"Simulation completed! Broke track at step {step}.")
            break

        elif len(delta_check) > 10:
            if show:
                print(f"Broke simulation at step {step}.")
            BreakCheck = True
            break
    
    if show:
        print(f'Broke simulation at step: {step} | Speed: {x_current[2]:.2f} m/s | Distance traveled: {s_total_traveled:.2f} / {track_length:.2f} m')


    # Convert historical data into numpy arrays for rendering 
    t_history = np.array(t_history)
    x_history = np.array(x_history)
    y_history = np.array(y_history)
    v_history = np.array(v_history)
    v_ref_history = np.array(v_ref_history)
    acc_history = np.array(acc_history)
    delta_history = np.array(delta_history)
    psi_history = np.array(psi_history)
    turning_history.append(1 if is_turning_sharp else 0)


    score = RMSE(v_ref_history, v_history)
    ySeries=[v_ref_history, v_history, acc_history, delta_history, turning_history]
    xSeries=[t_history for i in range(len(ySeries))]
    names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
            'Acceleration (m/s²)', 'Stirring Angle (deg)', 'Truning']
    data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]

    return score,BreakCheck,[xSeries,ySeries,names,data]


In [ ]:
score1,BreakCheck1,[xSeries1,ySeries1,names1,data1] = SimulateRT(
    dt=0.25,n_horizon = 20,sim_steps = 800,
    W_X = 1,W_Y = 1,W_speed = 10 ,W_acc = 1.5,W_delta = 0.25,W_U0 = 1,W_U1 = 2, size=1,show=True)

score2,BreakCheck2,[xSeries2,ySeries2,names2,data2] = SimulateRT(
    dt=0.25,n_horizon = 20,sim_steps = 800,
    W_X = 1,W_Y = 1,W_speed = 10 ,W_acc = 1.5,W_delta = 10.25,W_U0 = 1,W_U1 = 2, size=1,show=True)

print(score1,score2)

In [ ]:
i=1
PlotSeriesPLY(xSeries=[xSeries1[i],xSeries2[i]],
              ySeries=[ySeries1[i],ySeries2[i]],
              title=names1[i]
              )

In [ ]:
#PlotSeriesPLY(xSeries,ySeries,names,xLabel='Time (s)',title='Parameters Over Time')
PlotMPCTracksPLY(data1,width=800,height=500)


In [ ]:
n_horizon=6
dt = 0.5
sim_steps=800
size=1
show=True
W_X,W_Y,W_speed,W_acc,W_delta,W_U0,W_U1 = [1.0, 1.0, 9.37, 9.98, 2.42, 7.38, 9.25]

score,BreakCheck,[xSeries,ySeries,names,data] = SimulateRT(dt,n_horizon,sim_steps,
                                W_X,W_Y,W_speed,W_acc,W_delta,W_U0,W_U1,size,show)
PlotSeriesPLY(xSeries,ySeries,names,xLabel='Time (s)',title=f'Parameters Over Time|RMSE: {score}')
PlotMPCTracksPLY(data,width=800,height=500)